# Lab 2 — Confidence Scoring & Escalation

**Goal:** Make agents self-aware about uncertainty. An agent that knows it's guessing should say so — and route to humans automatically.

**Pattern:**
```
Agent generates answer + structured confidence score
    ↓
Score >= threshold  →  auto-proceed
Score <  threshold  →  escalate to human
Human approves / corrects / rejects
    ↓
Continue workflow
```

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()
print('Ready ✓')

## 1. Structured output with confidence score
Use Pydantic + OpenAI structured outputs so confidence is always a number, not free text.

In [ ]:
class AnswerWithConfidence(BaseModel):
    answer: str = Field(description='The answer to the question')
    confidence: float = Field(ge=0.0, le=1.0, description='0=no idea, 1=certain')
    reasoning: str = Field(description='One sentence explaining the confidence level')


def ask_with_confidence(question: str) -> AnswerWithConfidence:
    response = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Answer the question and give a confidence score. Be honest about uncertainty.'},
            {'role': 'user', 'content': question},
        ],
        response_format=AnswerWithConfidence,
    )
    return response.choices[0].message.parsed


# Test on easy and hard questions
questions = [
    'What is 2 + 2?',
    'What was the exact population of Tokyo on March 15, 2019?',
    'Is Python faster than C++?',
    'What will the S&P 500 close at tomorrow?',
]

for q in questions:
    result = ask_with_confidence(q)
    flag = '⚠️ ESCALATE' if result.confidence < 0.7 else '✅ auto'
    print(f'{flag} ({result.confidence:.0%}) {q}')
    print(f'   → {result.answer[:80]}')
    print(f'   Reasoning: {result.reasoning}')
    print()

## 2. Route to human when confidence is low

In [ ]:
from hitl import Step, ApprovalGate, ApprovalResult, Decision, HITLAgent

# Simulate escalation with callback
escalated_items = []

def escalation_callback(step: Step) -> ApprovalResult:
    escalated_items.append(step.name)
    print(f'  [HUMAN REVIEW] Step "{step.name}" confidence={step.confidence:.0%}')
    print(f'  Description: {step.description[:100]}')
    # In production: send to Slack, email, or UI. Here: auto-approve with note.
    return ApprovalResult(Decision.APPROVE, feedback='Reviewed and approved by simulated human.')


# Build steps from confidence scores
research_steps = []
for q in questions:
    result = ask_with_confidence(q)
    research_steps.append(Step(
        name=q[:30],
        description=f'Q: {q}\nA: {result.answer}',
        confidence=result.confidence,
        requires_approval=True,
    ))

gate = ApprovalGate(backend='callback', callback=escalation_callback, confidence_threshold=0.7)
agent = HITLAgent(gate=gate)
results = agent.run(research_steps, lambda step, ctx: step.description)

print(f'\n{len(escalated_items)} of {len(research_steps)} steps escalated to human: {escalated_items}')

## 3. Calibration check
A well-calibrated agent should be right ~70% of the time when it says 70% confident.

In [ ]:
# Simple calibration spot-check with known-answer trivia
trivia = [
    ('How many days in a leap year?', '366'),
    ('Capital of Australia?', 'Canberra'),
    ('Number of bones in adult human body?', '206'),
    ('Speed of light in m/s?', '299792458'),
]

correct = 0
confidences = []

for question, ground_truth in trivia:
    result = ask_with_confidence(question)
    is_correct = ground_truth.lower() in result.answer.lower()
    correct += is_correct
    confidences.append(result.confidence)
    symbol = '✓' if is_correct else '✗'
    print(f'{symbol} [{result.confidence:.0%}] {question} → {result.answer[:50]}')

avg_conf = sum(confidences) / len(confidences)
accuracy = correct / len(trivia)
print(f'\nAccuracy: {accuracy:.0%}  |  Avg confidence: {avg_conf:.0%}')
print(f'Calibration gap: {abs(accuracy - avg_conf):.0%} (smaller is better)')

## Exercise
Build a `ConfidenceRouter` that routes questions to one of three handlers:
- confidence ≥ 0.9 → answer directly
- 0.6 ≤ confidence < 0.9 → answer + add disclaimer
- confidence < 0.6 → refuse to answer, suggest where to look

Test it on at least 6 questions with varying difficulty.

In [ ]:
# Your code here
